# Mini-Challenge 1 — GroupBy: ward and DRG analytics
### Saint Mary's · Friday morning, 09:45

> *"Two questions: average length of stay per ward, and total revenue + average margin per DRG. Sorted, please."*

**Time:** 25 min · **Mode:** tandem · **Goal:** practise split-apply-combine.

---

# Solution — Mini-Challenge 1

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

HERE = Path.cwd()
DATA = next((p / "data" for p in [HERE, *HERE.parents] if (p / "data").exists()), None)
assert DATA is not None, "data/ folder not found"

enc = pd.read_csv(DATA / "encounters.csv", parse_dates=["admission_date","discharge_date"])
drg = pd.read_csv(DATA / "drg_catalog.csv")
patients = pd.read_csv(DATA / "patients.csv")

## Step 1 — Average LOS per ward

In [ ]:
los_by_ward = enc.groupby("ward")["los_days"].mean().sort_values(ascending=False).round(1)
print(los_by_ward)

## Step 2 — Revenue + margin per DRG

In [ ]:
enc = enc.merge(drg, on="drg_code", how="left")
enc["margin_eur"] = enc["base_reimbursement_eur"] - enc["actual_cost_eur"]

drg_summary = (enc.groupby(["drg_code","label"])
                  .agg(total_revenue=("base_reimbursement_eur","sum"),
                       mean_margin=("margin_eur","mean"),
                       n_cases=("encounter_id","count"))
                  .round(2)
                  .sort_values("mean_margin"))
print("--- Worst margin DRGs ---")
print(drg_summary.head(5))
print("--- Best margin DRGs ---")
print(drg_summary.tail(5))

## Fast-Track — Variance

In [ ]:
var_by_ward = enc.groupby("ward")["los_days"].std().sort_values(ascending=False).round(2)
print(var_by_ward)
# Interpretation: highest variance suggests the ward sees very mixed cases —
# from short observation stays to long, complicated stays.